# recheck with an LLM through Ollama (free T4)

E9: a small open model as the staleness judge (prediction-powered certificate, judge-directed
audits and the naive "ask the LLM" threshold), and as an agent deciding on its own whether to
re-check, with and without the certificate in its context. E12: the same judge on real PyPI
release traces, where the model's knowledge of how often a package releases can help.

- **Colab:** Runtime > Change runtime type > T4 GPU.
- **Kaggle:** Settings > Accelerator > GPU T4, and Settings > Internet on.

Every model call is cached in `cache/llm_cache.sqlite`, so a rerun (or `RECHECK_OFFLINE=1`) replays
it exactly. About an hour for both with `qwen2.5:3b`.

In [ ]:
# Get the code: clone it from GitHub, or upload recheck.zip to this session first.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/doraemonashish/recheck.git"   # <- your copy
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
os.chdir(WORK)
if not Path("recheck").exists():
    if Path("recheck.zip").exists() or Path("/kaggle/input").exists() and list(Path("/kaggle/input").rglob("recheck.zip")):
        z = Path("recheck.zip") if Path("recheck.zip").exists() else next(Path("/kaggle/input").rglob("recheck.zip"))
        subprocess.run(["unzip", "-q", str(z), "-d", str(WORK)], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO, "recheck"], check=True)
os.chdir(WORK / "recheck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("ready in", os.getcwd())

In [ ]:
# Install and start Ollama
import subprocess, time, os
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
server = subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
MODEL = os.environ.get("RECHECK_LLM_MODEL", "qwen2.5:3b")   # qwen2.5:7b also fits a T4
subprocess.run(["ollama", "pull", MODEL], check=True)
os.environ["RECHECK_LLM_MODEL"] = MODEL

In [ ]:
# Smoke test: one judge call on a fresh and an old belief
from recheck.agents.llm import LLMJudge
from recheck.types import Belief, Prov
j = LLMJudge()
for age in (0.1, 8.0):
    b = Belief("ticket.status|TCK-104", "in_review", 10.0 - age, Prov.OBSERVED)
    print(f"age {age} days -> P(stale) = {j(b, 10.0, None, 0, None)}")

In [ ]:
!recheck run configs/e9_llm_ollama.yaml

In [ ]:
from IPython.display import Markdown, Image, display
from pathlib import Path

def show(name):
    d = Path("results") / name
    display(Markdown((d / "README.md").read_text().split("![")[0]))
    for png in sorted(d.glob("*.png")):
        display(Image(filename=str(png), width=640))

show('e9_llm_ollama')

In [ ]:
!recheck run configs/e12_pypi_llm.yaml
show('e12_pypi_llm')

In [ ]:
# Keep the results. Kaggle keeps /kaggle/working; on Colab, download the zip from the file panel.
import subprocess
subprocess.run(["zip", "-qr", str(WORK / "recheck_results_ollama.zip"), "runs", "results", "cache"], check=False)
print("saved", WORK / "recheck_results_ollama.zip")